# 🥈 Notebook 02 — Silver Transformation

**Project:** Energy Fraud & Default Risk Detection  
**Layer:** Silver (Bronze → Cleansed + Typed + Joined)  
**Author:** Zara Louise  
**Stack:** PySpark + Delta Lake + Unity Catalog  
**Depends:** `01_bronze_ingestion` → `energy_project.bronze.*`

---

## 🎯 Goal

Read the three Bronze Delta tables and produce clean, typed, and enriched Silver tables — ready for Gold feature engineering and ML.

What happens in this layer:

- **Type casting** — strings → proper types (`Date`, `Double`, `Integer`)
- **Renaming** — PT-BR column names → English (data dictionary)
- **Cleaning** — trim whitespace, normalize nulls, drop exact duplicates
- **Validation** — quarantine rows that fail business rules
- **Enrichment** — join `inadimplencia` + `dominio_indicadores`
- **Persistence** — Silver Delta tables partitioned by `reference_year`

> ⚠️ **Spark best practice:** `.count()` is an action — expensive in big data (full scan). All transformations here are **lazy**. We trigger a single `.count()` per table only at the final validation cell.

---

## 📊 Tables produced

| Delta Table | Source (Bronze) | Approx. Rows |
|---|---|---|
| `energy_project.silver.samp` | `bronze.samp` | ~6.8M |
| `energy_project.silver.inadimplencia` | `bronze.inadimplencia` + `bronze.dominio_indicadores` | ~58 MB |
| `energy_project.silver.samp_quarantine` | rows failing quality rules | — |

---

## 🏛️ Medallion Architecture Context

- **Bronze** → Delta tables, all `StringType`, minimal transformation
- **Silver** (this layer) → Cleansed, typed, deduplicated, joined
- **Gold** → Business aggregations, ML features, BI-ready

In [0]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================

import pyspark.sql.functions as F
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType, DateType
)

print("✅ Libraries imported — ready for Silver transformation!")

✅ Libraries imported — ready for Silver transformation!


In [0]:
# =============================================================================
# PROJECT CONFIGURATION
# =============================================================================

CATALOG       = "energy_project"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

# Source tables (Bronze)
TABLE_B_SAMP    = f"{CATALOG}.{SCHEMA_BRONZE}.samp"
TABLE_B_INADIMP = f"{CATALOG}.{SCHEMA_BRONZE}.inadimplencia"
TABLE_B_DOMINIO = f"{CATALOG}.{SCHEMA_BRONZE}.dominio_indicadores"

# Destination tables (Silver)
TABLE_S_SAMP       = f"{CATALOG}.{SCHEMA_SILVER}.samp"
TABLE_S_INADIMP    = f"{CATALOG}.{SCHEMA_SILVER}.inadimplencia"
TABLE_S_QUARANTINE = f"{CATALOG}.{SCHEMA_SILVER}.samp_quarantine"

print("📂 Bronze sources:")
print(f"   {TABLE_B_SAMP}")
print(f"   {TABLE_B_INADIMP}")
print(f"   {TABLE_B_DOMINIO}")
print()
print("🎯 Silver destinations:")
print(f"   {TABLE_S_SAMP}")
print(f"   {TABLE_S_INADIMP}")
print(f"   {TABLE_S_QUARANTINE}")

📂 Bronze sources:
   energy_project.bronze.samp
   energy_project.bronze.inadimplencia
   energy_project.bronze.dominio_indicadores

🎯 Silver destinations:
   energy_project.silver.samp
   energy_project.silver.inadimplencia
   energy_project.silver.samp_quarantine


In [0]:
# =============================================================================
# SECTION 1 — READ BRONZE SAMP + INSPECT SCHEMA
# =============================================================================
# Before any casting, we confirm the exact column names coming from Bronze.
# printSchema() is a lazy operation — no full scan triggered here.

df_samp_bronze = spark.table(TABLE_B_SAMP)

print(f"📋 bronze.samp — schema ({len(df_samp_bronze.columns)} columns):")
df_samp_bronze.printSchema()

print("\n🔍 Sample row (vertical view):")
df_samp_bronze.show(1, vertical=True, truncate=False)

📋 bronze.samp — schema (22 columns):
root
 |-- DatGeracaoConjuntoDados: string (nullable = true)
 |-- NumCNPJAgenteDistribuidora: string (nullable = true)
 |-- SigAgenteDistribuidora: string (nullable = true)
 |-- NomAgenteDistribuidora: string (nullable = true)
 |-- NomTipoMercado: string (nullable = true)
 |-- DscModalidadeTarifaria: string (nullable = true)
 |-- DscSubGrupoTarifario: string (nullable = true)
 |-- DscClasseConsumoMercado: string (nullable = true)
 |-- DscSubClasseConsumidor: string (nullable = true)
 |-- DscDetalheConsumidor: string (nullable = true)
 |-- IdeAgenteAcessante: string (nullable = true)
 |-- NumCNPJAgenteAcessante: string (nullable = true)
 |-- NomAgenteAcessante: string (nullable = true)
 |-- DscPostoTarifario: string (nullable = true)
 |-- DscOpcaoEnergia: string (nullable = true)
 |-- DscDetalheMercado: string (nullable = true)
 |-- DatCompetencia: string (nullable = true)
 |-- VlrMercado: string (nullable = true)
 |-- _ingestion_timestamp: timestamp 

In [0]:
# =============================================================================
# SECTION 1 — SAMP: TYPE CASTING + RENAME (PT-BR → EN)
# =============================================================================
# Confirmed from printSchema():
#   - 18 business columns (all StringType)
#   - 4 audit columns (_ingestion_timestamp, _source_file, _ingestion_date, _partition_year)
#
# ⚠️  VlrMercado uses comma as decimal separator ("8,000000")
#     → must replace "," with "." before casting to Double

df_samp_typed = (
    df_samp_bronze

    # ── Dates ────────────────────────────────────────────────────────────────
    .withColumn("dataset_generation_date",
        F.to_date(F.col("DatGeracaoConjuntoDados"), "yyyy-MM-dd"))

    .withColumn("reference_date",
        F.to_date(F.col("DatCompetencia"), "yyyy-MM-dd"))

    # ── Numeric (fix decimal separator before cast) ───────────────────────────
    .withColumn("market_value",
        F.regexp_replace(F.col("VlrMercado"), ",", ".").cast(DoubleType()))

    # ── Strings (trim whitespace) ─────────────────────────────────────────────
    .withColumn("distributor_cnpj",     F.trim(F.col("NumCNPJAgenteDistribuidora")))
    .withColumn("distributor_code",     F.trim(F.col("SigAgenteDistribuidora")))
    .withColumn("distributor_name",     F.trim(F.col("NomAgenteDistribuidora")))
    .withColumn("market_type",          F.trim(F.col("NomTipoMercado")))
    .withColumn("tariff_modality",      F.trim(F.col("DscModalidadeTarifaria")))
    .withColumn("tariff_subgroup",      F.trim(F.col("DscSubGrupoTarifario")))
    .withColumn("consumption_class",    F.trim(F.col("DscClasseConsumoMercado")))
    .withColumn("consumer_subclass",    F.trim(F.col("DscSubClasseConsumidor")))
    .withColumn("consumer_detail",      F.trim(F.col("DscDetalheConsumidor")))
    .withColumn("accessing_agent_id",   F.trim(F.col("IdeAgenteAcessante")))
    .withColumn("accessing_agent_cnpj", F.trim(F.col("NumCNPJAgenteAcessante")))
    .withColumn("accessing_agent_name", F.trim(F.col("NomAgenteAcessante")))
    .withColumn("tariff_period",        F.trim(F.col("DscPostoTarifario")))
    .withColumn("energy_option",        F.trim(F.col("DscOpcaoEnergia")))
    .withColumn("market_detail",        F.trim(F.col("DscDetalheMercado")))

    # ── Derived ───────────────────────────────────────────────────────────────
    .withColumn("reference_year",  F.year(F.col("reference_date")))
    .withColumn("reference_month", F.month(F.col("reference_date")))

    # ── Select final columns (drop original PT-BR, keep audit) ────────────────
    .select(
        "dataset_generation_date",
        "distributor_cnpj",
        "distributor_code",
        "distributor_name",
        "market_type",
        "tariff_modality",
        "tariff_subgroup",
        "consumption_class",
        "consumer_subclass",
        "consumer_detail",
        "accessing_agent_id",
        "accessing_agent_cnpj",
        "accessing_agent_name",
        "tariff_period",
        "energy_option",
        "market_detail",
        "reference_date",
        "market_value",
        "reference_year",
        "reference_month",
        # audit (from Bronze)
        "_ingestion_timestamp",
        "_source_file",
        "_ingestion_date",
    )
)

print("✅ Cast + rename applied (lazy — not yet executed)")
print(f"   Columns: {len(df_samp_typed.columns)}")
print()
print("📋 New schema:")
df_samp_typed.printSchema()

✅ Cast + rename applied (lazy — not yet executed)
   Columns: 23

📋 New schema:
root
 |-- dataset_generation_date: date (nullable = true)
 |-- distributor_cnpj: string (nullable = true)
 |-- distributor_code: string (nullable = true)
 |-- distributor_name: string (nullable = true)
 |-- market_type: string (nullable = true)
 |-- tariff_modality: string (nullable = true)
 |-- tariff_subgroup: string (nullable = true)
 |-- consumption_class: string (nullable = true)
 |-- consumer_subclass: string (nullable = true)
 |-- consumer_detail: string (nullable = true)
 |-- accessing_agent_id: string (nullable = true)
 |-- accessing_agent_cnpj: string (nullable = true)
 |-- accessing_agent_name: string (nullable = true)
 |-- tariff_period: string (nullable = true)
 |-- energy_option: string (nullable = true)
 |-- market_detail: string (nullable = true)
 |-- reference_date: date (nullable = true)
 |-- market_value: double (nullable = true)
 |-- reference_year: integer (nullable = true)
 |-- referen

In [0]:
# =============================================================================
# SECTION 1 — SAMP: DATA QUALITY + DEDUPLICATION
# =============================================================================
# Quality rules (rows failing go to quarantine):
#   • reference_date   → null means cast failed (unparseable date)
#   • market_value     → null means cast failed (bad numeric format)
#
# Dedup key: unique combination that identifies a single market measurement.
#
# ⚠️  filter() and dropDuplicates() are transformations (lazy).
#     No action triggered here.

# ── Split valid / quarantine ──────────────────────────────────────────────────
df_samp_valid = df_samp_typed.filter(
    F.col("reference_date").isNotNull() &
    F.col("market_value").isNotNull()
)

df_samp_quarantine = df_samp_typed.filter(
    F.col("reference_date").isNull() |
    F.col("market_value").isNull()
)

# ── Deduplication on business key ────────────────────────────────────────────
SAMP_DEDUP_KEYS = [
    "distributor_cnpj",
    "reference_date",
    "consumption_class",
    "tariff_modality",
    "tariff_subgroup",
    "market_detail",
    "market_value",
]

df_samp_silver = df_samp_valid.dropDuplicates(SAMP_DEDUP_KEYS)

print("✅ Quality filters defined (lazy):")
print("   • df_samp_valid      → rows passing all quality rules")
print("   • df_samp_quarantine → rows with null reference_date or market_value")
print()
print("✅ Deduplication rule applied (lazy):")
print(f"   Business key: {SAMP_DEDUP_KEYS}")

✅ Quality filters defined (lazy):
   • df_samp_valid      → rows passing all quality rules
   • df_samp_quarantine → rows with null reference_date or market_value

✅ Deduplication rule applied (lazy):
   Business key: ['distributor_cnpj', 'reference_date', 'consumption_class', 'tariff_modality', 'tariff_subgroup', 'market_detail', 'market_value']


In [0]:
# =============================================================================
# SECTION 1 — SAMP: WRITE TO SILVER
# =============================================================================
# This is the first action of the notebook — triggers the full DAG:
#   read Bronze → cast → filter → dedup → write Silver
#
# Partition by reference_year (same strategy as Bronze).
# Quarantine table written separately for investigation.
#
# ⏳ Expected time: 2-4 minutes for ~1.5 GB

print(f"💾 Writing silver.samp → {TABLE_S_SAMP} ...")

(
    df_samp_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("reference_year")
    .saveAsTable(TABLE_S_SAMP)
)

print(f"✅ silver.samp written successfully!")
print()
print(f"💾 Writing silver.samp_quarantine → {TABLE_S_QUARANTINE} ...")

(
    df_samp_quarantine.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_S_QUARANTINE)
)

print(f"✅ silver.samp_quarantine written successfully!")

💾 Writing silver.samp → energy_project.silver.samp ...
✅ silver.samp written successfully!

💾 Writing silver.samp_quarantine → energy_project.silver.samp_quarantine ...
✅ silver.samp_quarantine written successfully!


In [0]:
# =============================================================================
# SECTION 1 — SAMP: VALIDATION
# =============================================================================
# Single .count() per table — triggered here only, not mid-pipeline.
# Re-reads from Unity Catalog to confirm the table is queryable.

df_s_samp     = spark.table(TABLE_S_SAMP)
df_s_quarant  = spark.table(TABLE_S_QUARANTINE)

count_samp    = df_s_samp.count()
count_quarant = df_s_quarant.count()

print("=" * 60)
print("🔍 VALIDATION — silver.samp")
print("=" * 60)

print(f"\n📊 Row counts:")
print(f"   silver.samp            → {count_samp:>12,} rows")
print(f"   silver.samp_quarantine → {count_quarant:>12,} rows  ← should be 0 or very small")

print(f"\n🗂️  Partitions (reference_year):")
df_s_samp.select("reference_year").distinct().orderBy("reference_year").show()

print(f"\n📅 reference_date range:")
df_s_samp.agg(
    F.min("reference_date").alias("min"),
    F.max("reference_date").alias("max")
).show()

print(f"\n❓ Null check on critical columns:")
df_s_samp.select(
    F.count(F.when(F.col("reference_date").isNull(),  1)).alias("null_reference_date"),
    F.count(F.when(F.col("market_value").isNull(),    1)).alias("null_market_value"),
    F.count(F.when(F.col("distributor_cnpj").isNull(),1)).alias("null_distributor_cnpj"),
).show()

print(f"\n🔍 Sample rows:")
df_s_samp.select(
    "reference_date", "distributor_code",
    "consumption_class", "market_value", "reference_year"
).show(5, truncate=False)

print("=" * 60)
print(f"✅ silver.samp validation complete!")
print("=" * 60)

🔍 VALIDATION — silver.samp

📊 Row counts:
   silver.samp            →    6,319,615 rows
   silver.samp_quarantine →            0 rows  ← should be 0 or very small

🗂️  Partitions (reference_year):
+--------------+
|reference_year|
+--------------+
|          2020|
|          2021|
|          2022|
|          2023|
|          2024|
|          2025|
|          2026|
+--------------+


📅 reference_date range:
+----------+----------+
|       min|       max|
+----------+----------+
|2020-01-01|2026-03-01|
+----------+----------+


❓ Null check on critical columns:
+-------------------+-----------------+---------------------+
|null_reference_date|null_market_value|null_distributor_cnpj|
+-------------------+-----------------+---------------------+
|                  0|                0|                    0|
+-------------------+-----------------+---------------------+


🔍 Sample rows:
+--------------+----------------+------------------+------------+--------------+
|reference_date|distributo

In [0]:
# =============================================================================
# SECTION 2 — READ BRONZE INADIMPLENCIA + DOMINIO + INSPECT
# =============================================================================

df_inadimp_bronze = spark.table(TABLE_B_INADIMP)
df_dominio_bronze = spark.table(TABLE_B_DOMINIO)

print(f"📋 bronze.inadimplencia — schema ({len(df_inadimp_bronze.columns)} columns):")
df_inadimp_bronze.printSchema()

print(f"\n📋 bronze.dominio_indicadores — schema ({len(df_dominio_bronze.columns)} columns):")
df_dominio_bronze.printSchema()

📋 bronze.inadimplencia — schema (10 columns):
root
 |-- DatGeracaoConjuntoDados: string (nullable = true)
 |-- SigAgente: string (nullable = true)
 |-- NumCNPJ: string (nullable = true)
 |-- SigIndicador: string (nullable = true)
 |-- AnoIndice: string (nullable = true)
 |-- NumPeriodoIndice: string (nullable = true)
 |-- VlrIndiceEnviado: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_date: date (nullable = true)


📋 bronze.dominio_indicadores — schema (6 columns):
root
 |-- DatGeracaoConjuntoDados: string (nullable = true)
 |-- SigIndicador: string (nullable = true)
 |-- DscIndicador: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_date: date (nullable = true)



In [0]:
# =============================================================================
# SECTION 2 — INADIMPLENCIA: CAST + RENAME + JOIN DOMINIO
# =============================================================================
# Confirmed columns from Bronze:
#   DatGeracaoConjuntoDados → dataset_generation_date  (Date, format dd-MM-yyyy)
#   SigAgente               → distributor_code         (String)
#   NumCNPJ                 → distributor_cnpj         (String)
#   SigIndicador            → indicator_code           (String — FK to dominio)
#   AnoIndice               → reference_year           (Integer)
#   NumPeriodoIndice        → reference_month          (Integer)
#   VlrIndiceEnviado        → indicator_value          (Double, comma as decimal)
#
# ⚠️  No single reference_date column exists — we BUILD it from:
#     AnoIndice + NumPeriodoIndice → first day of that month
#     e.g. AnoIndice=2012, NumPeriodoIndice=1 → reference_date=2012-01-01

# ── Prepare dominio (small dimension table) ───────────────────────────────────
df_dominio = (
    df_dominio_bronze
    .withColumn("indicator_code",        F.trim(F.col("SigIndicador")))
    .withColumn("indicator_description", F.trim(F.col("DscIndicador")))
    .select("indicator_code", "indicator_description")
    .dropDuplicates(["indicator_code"])
)

# ── Cast + rename inadimplencia ───────────────────────────────────────────────
df_inadimp_typed = (
    df_inadimp_bronze

    # ── Dates ─────────────────────────────────────────────────────────────────
    .withColumn("dataset_generation_date",
        F.to_date(F.col("DatGeracaoConjuntoDados"), "dd-MM-yyyy"))

    # ── Build reference_date from year + month ────────────────────────────────
    .withColumn("reference_year",  F.col("AnoIndice").cast("integer"))
    .withColumn("reference_month", F.col("NumPeriodoIndice").cast("integer"))
    .withColumn("reference_date",
        F.to_date(
            F.concat_ws("-",
                F.col("AnoIndice"),
                F.lpad(F.col("NumPeriodoIndice"), 2, "0"),
                F.lit("01")
            ),
            "yyyy-MM-dd"
        )
    )

    # ── Numeric ───────────────────────────────────────────────────────────────
    .withColumn("indicator_value",
        F.regexp_replace(F.col("VlrIndiceEnviado"), ",", ".").cast("double"))

    # ── Strings ───────────────────────────────────────────────────────────────
    .withColumn("distributor_code", F.trim(F.col("SigAgente")))
    .withColumn("distributor_cnpj", F.trim(F.col("NumCNPJ")))
    .withColumn("indicator_code",   F.trim(F.col("SigIndicador")))
)

# ── Broadcast join with dominio ───────────────────────────────────────────────
df_inadimp_enriched = (
    df_inadimp_typed
    .join(F.broadcast(df_dominio), on="indicator_code", how="left")
)

# ── Final select ──────────────────────────────────────────────────────────────
df_inadimp_silver = (
    df_inadimp_enriched
    .filter(
        F.col("reference_date").isNotNull() &
        F.col("indicator_value").isNotNull()
    )
    .dropDuplicates(["distributor_cnpj", "indicator_code", "reference_date"])
    .select(
        "dataset_generation_date",
        "distributor_cnpj",
        "distributor_code",
        "indicator_code",
        "indicator_description",
        "reference_date",
        "reference_year",
        "reference_month",
        "indicator_value",
        "_ingestion_timestamp",
        "_source_file",
        "_ingestion_date",
    )
)

print("✅ Cast + rename + join applied (lazy)")
print(f"\n📋 New schema:")
df_inadimp_silver.printSchema()

✅ Cast + rename + join applied (lazy)

📋 New schema:
root
 |-- dataset_generation_date: date (nullable = true)
 |-- distributor_cnpj: string (nullable = true)
 |-- distributor_code: string (nullable = true)
 |-- indicator_code: string (nullable = true)
 |-- indicator_description: string (nullable = true)
 |-- reference_date: date (nullable = false)
 |-- reference_year: integer (nullable = true)
 |-- reference_month: integer (nullable = true)
 |-- indicator_value: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_date: date (nullable = true)



In [0]:
# =============================================================================
# SECTION 2 — INADIMPLENCIA: WRITE TO SILVER
# =============================================================================

print(f"💾 Writing to {TABLE_S_INADIMP} ...")

(
    df_inadimp_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("reference_year")
    .saveAsTable(TABLE_S_INADIMP)
)

print(f"✅ silver.inadimplencia written successfully!")

💾 Writing to energy_project.silver.inadimplencia ...
✅ silver.inadimplencia written successfully!


In [0]:
# =============================================================================
# SECTION 2 — INADIMPLENCIA: VALIDATION
# =============================================================================

df_s_inadimp = spark.table(TABLE_S_INADIMP)
count_inadimp = df_s_inadimp.count()

print("=" * 60)
print("🔍 VALIDATION — silver.inadimplencia")
print("=" * 60)

print(f"\n📊 Row count: {count_inadimp:,}")

print(f"\n🗂️  Partitions (reference_year):")
df_s_inadimp.select("reference_year").distinct().orderBy("reference_year").show()

print(f"\n📅 reference_date range:")
df_s_inadimp.agg(
    F.min("reference_date").alias("min"),
    F.max("reference_date").alias("max")
).show()

print(f"\n❓ Null check:")
df_s_inadimp.select(
    F.count(F.when(F.col("reference_date").isNull(),     1)).alias("null_reference_date"),
    F.count(F.when(F.col("indicator_value").isNull(),    1)).alias("null_indicator_value"),
    F.count(F.when(F.col("indicator_description").isNull(), 1)).alias("null_indicator_desc"),
).show()

print(f"\n🔍 Sample rows:")
df_s_inadimp.select(
    "reference_date", "distributor_code",
    "indicator_code", "indicator_description",
    "indicator_value"
).show(5, truncate=False)

print("=" * 60)
print(f"✅ silver.inadimplencia validation complete!")
print("=" * 60)

🔍 VALIDATION — silver.inadimplencia

📊 Row count: 1,104,153

🗂️  Partitions (reference_year):
+--------------+
|reference_year|
+--------------+
|          2012|
|          2013|
|          2014|
|          2015|
|          2016|
|          2017|
|          2018|
|          2019|
|          2020|
|          2021|
|          2022|
|          2023|
|          2024|
|          2025|
|          2026|
+--------------+


📅 reference_date range:
+----------+----------+
|       min|       max|
+----------+----------+
|2012-01-01|2026-03-01|
+----------+----------+


❓ Null check:
+-------------------+--------------------+-------------------+
|null_reference_date|null_indicator_value|null_indicator_desc|
+-------------------+--------------------+-------------------+
|                  0|                   0|                  0|
+-------------------+--------------------+-------------------+


🔍 Sample rows:
+--------------+----------------+--------------+-----------------------------------------